# 06 — Sidechain compression: GR computed on one signal, applied to another

**No training.** The stated end-goal of the two-stage decomposition
(`MODEL_GAIN_PRIOR.md` §6): because the interface between the stages is a *physical
quantity* (GR in dB) rather than a learned latent, the detector can be pointed at a
**different signal** than the one being processed:

```
sidechain source ──► DetectorGRLSTM ──► GR (dB) ──► GainPriorDiffSSLLSTM ──► ducked programme
programme signal ────────────────────────────────────────────────▲
```

A monolithic end-to-end compressor model has no place to inject an external
sidechain — this capability exists *only* because of the factorisation, so even a
qualitative demonstration is an architectural result. There is no ground truth
(the hardware was never recorded in sidechain mode); the evaluation is behavioural:
does the programme audibly and measurably duck when — and only when — the sidechain
is active, with knob-consistent depth and ballistics?

Two sidechain sources: another song's dry recording (radio-style ducking), and a
synthetic kick-drum pattern (the classic EDM pump). Audio is exported to
`eval_output/06_sidechain/`.

In [1]:
# -- 0. Setup ------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import torch
from IPython.display import Audio, display

from exp_common import (
    SR, db_to_amp, ensure_eval_out, knobs, load_detector, load_gain_prior,
    load_pair, load_split, pairs_from_keys, params_for, short_env_db,
    stream_detector_gr, stream_gain_prior,
)

SELECTED_GAIN_PRIOR_RUN = "gain_prior_20260702_085618_diffssl_lstm32_gain_prior"
SELECTED_DETECTOR_RUN = "lstm_gr_20260702_174039_lstm_detector_gr"

MODEL, HP, RUN_DIR = load_gain_prior(SELECTED_GAIN_PRIOR_RUN)
DET, DET_HP, DET_DIR = load_detector(SELECTED_DETECTOR_RUN)
SPLIT = load_split(RUN_DIR)
VAL_PAIRS = pairs_from_keys(SPLIT.val_pair_keys)
TEST_PAIRS = pairs_from_keys(SPLIT.test_pair_keys)
OUT = ensure_eval_out() / "06_sidechain"
OUT.mkdir(exist_ok=True)

DUR_SEC = 30.0
# heavy pump setting: deep threshold, fast-ish attack, musical release, high ratio
K = knobs(-12.0, 3.0, 0.4, 10.0)

# programme: a validation song excerpt; music sidechain: a held-out TEST song
prog_song, prog_setting = VAL_PAIRS[0]
sc_song, sc_setting = TEST_PAIRS[0]
prog, _, _ = load_pair(prog_setting, prog_song, start_sec=30.0, duration_sec=DUR_SEC)
sc_music, _, _ = load_pair(sc_setting, sc_song, start_sec=60.0, duration_sec=DUR_SEC)
T = min(prog.shape[-1], sc_music.shape[-1])
prog, sc_music = prog[..., :T], sc_music[..., :T]
print(f"programme: {prog_song} | music sidechain: {sc_song} | {T/SR:.0f} s")

/Volumes/Saola's Drive/AllCode/thesis/Virtual-Analogue-Compressor-Modelling/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 8,322-param GainPriorDiffSSLLSTM  (gain_prior_20260702_085618_diffssl_lstm32_gain_prior / best-059-730800.ckpt)


RuntimeError: Error(s) in loading state_dict for DetectorGRLSTM:
	Missing key(s) in state_dict: "film.weight", "film.bias", "glu.weight", "glu.bias". 
	size mismatch for lstm.weight_ih_l0: copying a param with shape torch.Size([128, 8]) from checkpoint, the shape in current model is torch.Size([128, 4]).

In [ ]:
# -- 1. Synthetic kick-pattern sidechain (the classic pump source) --------------
def kick_pattern(dur_sec, bpm=120.0, f0=60.0, decay_s=0.08, hit_len_s=0.35,
                 peak_dbfs=-3.0, sr=SR):
    x = torch.zeros(int(dur_sec * sr))
    period = int(60.0 / bpm * sr)
    hit_t = torch.arange(int(hit_len_s * sr), dtype=torch.float32) / sr
    # pitch-sweeping decaying sine - a plausible kick, still just a level pattern
    hit = torch.sin(2 * np.pi * (f0 * hit_t + 30.0 * decay_s * (1 - torch.exp(-hit_t / decay_s))))
    hit = hit * torch.exp(-hit_t / decay_s) * db_to_amp(peak_dbfs)
    for o in range(0, len(x) - len(hit), period):
        x[o:o + len(hit)] += hit
    return x.unsqueeze(0)

sc_kick = kick_pattern(T / SR)[..., :T]
print(f"kick sidechain: {sc_kick.shape[-1]/SR:.0f} s @ 120 BPM")

In [ ]:
# -- 2. Run the sidechain cascade ------------------------------------------------
# GR is predicted from the SIDECHAIN, applied to the PROGRAMME. The self-GR run
# (GR from the programme itself) is the normal-compression reference.

@torch.no_grad()
def sidechain_compress(programme, sidechain, k):
    gr = stream_detector_gr(DET, sidechain, k, sample_len=programme.shape[-1])
    out = stream_gain_prior(MODEL, programme, gr, k)
    return out, gr

ducked_music, gr_music = sidechain_compress(prog, sc_music, K)
ducked_kick, gr_kick = sidechain_compress(prog, sc_kick, K)
selfcomp, gr_self = sidechain_compress(prog, prog, K)
print("mean GR - music sidechain: %.2f dB | kick sidechain: %.2f dB | self: %.2f dB"
      % (gr_music.mean(), gr_kick.mean(), gr_self.mean()))

In [ ]:
# -- 3. Behavioural evidence: envelopes + GR, kick case zoomed -------------------
a, b = int(8.0 * SR), int(14.0 * SR)          # 6 s zoom
t = np.arange(a, b) / SR

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
axes[0].plot(t, sc_kick[0, a:b].numpy(), lw=0.5, color="#444")
axes[0].set_ylabel("sidechain"); axes[0].set_title("Kick-pattern sidechain source")
axes[0].grid(alpha=0.3)

axes[1].plot(t, gr_kick[0, a:b].numpy(), lw=1.0, color="#d62728")
axes[1].set_ylabel("predicted GR (dB)"); axes[1].grid(alpha=0.3)
axes[1].set_title("GR predicted FROM the sidechain (knobs: thr -12, atk 3 ms, rel 0.4 s, ratio 10)")

env_before = short_env_db(prog[..., a:b], 1024)
env_after = short_env_db(ducked_kick[..., a:b], 1024)
axes[2].plot(t, env_before, lw=1.0, color="#1f77b4", label="programme before")
axes[2].plot(t, env_after, lw=1.0, color="#d62728", alpha=0.85, label="programme after (ducked)")
axes[2].plot(t, env_after - env_before, lw=0.8, color="#2ca02c", alpha=0.8,
             label="applied gain (after - before)")
axes[2].set_xlabel("time (s)"); axes[2].set_ylabel("envelope (dB)")
axes[2].set_title("Programme ducking, time-locked to a signal it never 'hears'")
axes[2].grid(alpha=0.3); axes[2].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT.parent / "06_sidechain_kick_zoom.png", dpi=150, bbox_inches="tight")
plt.show()

# quantify: correlation between sidechain envelope and applied gain
sc_env = short_env_db(sc_kick, 1024)
applied = short_env_db(ducked_kick, 1024) - short_env_db(prog, 1024)
m = sc_env > -70
print(f"corr(sidechain envelope, applied gain) = "
      f"{np.corrcoef(sc_env[m], applied[m])[0, 1]:.3f}  (should be strongly negative)")

In [ ]:
# -- 4. Music sidechain overview + all three GR curves ---------------------------
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True)
tt = np.arange(T) / SR
ax1.plot(tt, short_env_db(sc_music, 4096), lw=0.9, color="#444", label="music sidechain env")
ax1.plot(tt, short_env_db(prog, 4096), lw=0.9, color="#1f77b4", alpha=0.7, label="programme env")
ax1.set_ylabel("envelope (dB)"); ax1.grid(alpha=0.3); ax1.legend(fontsize=8)
ax1.set_title("Radio-style ducking: speech/music sidechain vs programme")

ax2.plot(tt, gr_music[0].numpy(), lw=0.8, color="#d62728", label="GR from music sidechain")
ax2.plot(tt, gr_kick[0].numpy(), lw=0.6, color="#ff7f0e", alpha=0.7, label="GR from kick pattern")
ax2.plot(tt, gr_self[0].numpy(), lw=0.6, color="#1f77b4", alpha=0.7, label="GR from programme (self)")
ax2.set_xlabel("time (s)"); ax2.set_ylabel("GR (dB)")
ax2.grid(alpha=0.3); ax2.legend(fontsize=8)
ax2.set_title("Same programme, three conditioning signals - the interface is substitutable")
fig.tight_layout()
fig.savefig(OUT.parent / "06_sidechain_overview.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- 5. Export audio + listen -----------------------------------------------------
def save(name, x):
    path = OUT / f"{name}.wav"
    sf.write(path, x[0].numpy(), SR)
    print(f"wrote {path}")
    return path

save("programme_dry", prog)
save("sidechain_music", sc_music)
save("sidechain_kick", sc_kick)
save("ducked_by_music", ducked_music)
save("ducked_by_kick", ducked_kick)
save("self_compressed", selfcomp)

# stereo audition: L = ducked programme, R = sidechain (hear the time-lock)
audition = torch.stack([ducked_kick[0], sc_kick[0] * 0.5], dim=1).numpy()
sf.write(OUT / "audition_ducked_L_kick_R.wav", audition, SR)

print("\nprogramme (dry):");            display(Audio(prog[0].numpy(), rate=SR))
print("ducked by kick pattern:");       display(Audio(ducked_kick[0].numpy(), rate=SR))
print("ducked by music sidechain:");    display(Audio(ducked_music[0].numpy(), rate=SR))
print("self-compressed (reference):");  display(Audio(selfcomp[0].numpy(), rate=SR))

## Reading the results

- The **applied-gain trace** (green) locking to the kick pattern — a signal the
  gain-application stage never receives as audio — is the demonstrable capability:
  conditioning by an *external physical control signal*, not feature extraction from
  the input. The correlation number quantifies it.
- **GR-curve overlay**: identical programme, three different gain trajectories —
  the substitutability claim of `MODEL_DETECTOR_GR.md` §6.2 in one figure.
- Caveats to state in the thesis: the kick pattern is out-of-distribution for the
  detector *as content* (it was trained on multitrack music) but in-distribution *as
  level trajectory* — its frontend reduces everything to energy envelopes by
  construction (notebook 03 measures exactly this invariance). No ground truth
  exists for the sidechain mode; the claim is capability + plausibility (knob-true
  depth/ballistics per notebook 03), not accuracy.